In [1]:
import decimal
import json
import logging
from typing import Any, Dict, List

import boto3
from botocore.exceptions import ClientError
from IPython.display import HTML

In [ ]:
class DynamoDBManager:
    def __init__(self, region="us-east-1"):
        self.dynamodb = boto3.client("dynamodb", region_name=region)

    def create_table(self, table_name, key_schema, attribute_definitions, provisioned_throughput):
        try:
            response = self.dynamodb.create_table(
                TableName=table_name,
                KeySchema=key_schema,
                AttributeDefinitions=attribute_definitions,
                ProvisionedThroughput=provisioned_throughput
            )

            print(f"Creating table: {table_name}")

            # Wait until table exists
            waiter = self.dynamodb.get_waiter("table_exists")
            waiter.wait(TableName=table_name)

            print(f"Table created: {table_name}")

        except ClientError as e:
            print("Error creating table:", e)

In [2]:
with open("table_property.json") as f:
    config = json.load(f)
manager = DynamoDBManager()
for table in config["tables"]:
    manager.create_table(
        table_name=table["table_name"],
        key_schema=table["kwargs"]["KeySchema"],
        attribute_definitions=table["kwargs"]["AttributeDefinitions"],
        provisioned_throughput=table["kwargs"]["ProvisionedThroughput"]
    )

NameError: name 'DynamoDBManager' is not defined

In [4]:
def read_data(file_path: str) -> Dict[str, Any]:
    with open(file_path, "r") as json_file:
        items = json.load(json_file)
    return items

In [6]:
read_data("data/Forum.json")

{'Forum': [{'PutRequest': {'Item': {'Name': {'S': 'Amazon DynamoDB'},
     'Category': {'S': 'Amazon Web Services'},
     'Threads': {'N': '2'},
     'Messages': {'N': '4'},
     'Views': {'N': '1000'}}}},
  {'PutRequest': {'Item': {'Name': {'S': 'Amazon S3'},
     'Category': {'S': 'Amazon Web Services'}}}}]}

In [27]:
def load_data_from_json(config_file):
       # try:
            client = boto3.client("dynamodb")
            

            with open(config_file, "r") as f:
                config = json.load(f)
            
            if isinstance(config, dict):
                items = list(config.values())[0]   # get the first list
            else:
                items = config

            for table_config in config["tables"]:
                table_name = table_config["table_name"]
                file_path = table_config["file"]

                print(f"\nLoading data into {table_name} from {file_path}")
                with open(file_path, "r") as json_file:
                    items = json.load(json_file)
                    item=items[table_name.split('-')[-1]]
                    #print(f'{item}')
                
                    client.put_item(TableName=table_name, Item=item[0]['PutRequest']['Item'])
                   # print(f"{item[0]['PutRequest']['Item']} in {table_name}")





                

       # except ClientError as e:
        #    print("AWS Error:", e)
       # except Exception as e:
         #   print("Error:", e)

In [28]:
load_data_from_json("data_config.json")


Loading data into mycourse-ProductCatalog from data/ProductCatalog.json

Loading data into mycourse-Forum from data/Forum.json

Loading data into mycourse-Thread from data/Thread.json

Loading data into mycourse-Reply from data/Reply.json


In [51]:
def batch_write_item_db(config_file):
       # try:
            
            client = boto3.client("dynamodb")

            with open(config_file, "r") as f:
                config = json.load(f)
            

            for table_config in config["tables"]:
                table_name = table_config["table_name"]
                file_path = table_config["file"]

                print(f"\nLoading data into {table_name} from {file_path}")
                with open(file_path, "r") as json_file:
                    items = json.load(json_file)
                   
                
                    client.batch_write_item( RequestItems=items)
                    print(f"{items} in {table_name}")





                

       # except ClientError as e:
        #    print("AWS Error:", e)
       # except Exception as e:
         #   print("Error:", e)

In [52]:
batch_write_item_db("data_config.json")


Loading data into ProductCatalog from data/ProductCatalog.json
{'ProductCatalog': [{'PutRequest': {'Item': {'Id': {'N': '101'}, 'Title': {'S': 'Book 101 Title'}, 'ISBN': {'S': '111-1111111111'}, 'Authors': {'L': [{'S': 'Author1'}]}, 'Price': {'N': '2'}, 'Dimensions': {'S': '8.5 x 11.0 x 0.5'}, 'PageCount': {'N': '500'}, 'InPublication': {'BOOL': True}, 'ProductCategory': {'S': 'Book'}}}}, {'PutRequest': {'Item': {'Id': {'N': '102'}, 'Title': {'S': 'Book 102 Title'}, 'ISBN': {'S': '222-2222222222'}, 'Authors': {'L': [{'S': 'Author1'}, {'S': 'Author2'}]}, 'Price': {'N': '20'}, 'Dimensions': {'S': '8.5 x 11.0 x 0.8'}, 'PageCount': {'N': '600'}, 'InPublication': {'BOOL': True}, 'ProductCategory': {'S': 'Book'}}}}, {'PutRequest': {'Item': {'Id': {'N': '103'}, 'Title': {'S': 'Book 103 Title'}, 'ISBN': {'S': '333-3333333333'}, 'Authors': {'L': [{'S': 'Author1'}, {'S': 'Author2'}]}, 'Price': {'N': '2000'}, 'Dimensions': {'S': '8.5 x 11.0 x 1.5'}, 'PageCount': {'N': '600'}, 'InPublication': {'

ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the BatchWriteItem operation: Requested resource not found